In [5]:
from pathlib import Path

import numpy as np
import scipy.optimize as spo

from cardiac_electrophysiology import posterior_builder as builder
from cardiac_electrophysiology.utils import analysis, visualization

In [36]:
settings = builder.PosteriorBuilderSettings(
    paths=builder.Paths(
        mesh_path=Path("../data/mesh.vtu"),
        basis_vecs_path=Path("../data/basis_vecs.npy"),
        log_file_path=Path("../results/lsbip_logfile.log"),
        prior_mean_path=Path("../data/prior_mean_from_sde.npy"),
        ground_truth_path=Path("../data/ground_truth_from_sde.npy"),
    ),
    prior_parameters=builder.PriorParameters(
        kappa=1,
        tau=0.1,
        seed=42,
    ),
    eikonal_parameters=builder.EikonalParameters(
        solver_tolerance=1e-6,
        max_num_iterations=1000,
        max_value=1000,
        initial_site_ind=12650,
        longitudinal_velocity=3,
        transversal_velocity=1,
    ),
    observation_parameters=builder.ObservationParameters(
        num_observations=1000,
        noise_variance=1e-4,
        seed=0,
    ),
    logger_settings=builder.LoggerSettings(
        do_printing=False,
        write_mode="w",
    ),
)

In [40]:
test_vec = np.load("test_vec.npy")

In [41]:
posterior_builder = builder.PosteriorBuilder(settings)
posterior, additional_output = posterior_builder.build(return_additional_data=True)

In [42]:
posterior.prior.evaluate_cost(test_vec)

np.float64(297704.49431993836)

In [16]:
optimizer_options = {
    "disp": True,
    "maxiter": 1000,
    "ftol": 1e-3,
    "gtol": 1e-3,
    "maxls": 100,
}
initial_guess = np.zeros(additional_output.pv_mesh.number_of_points)
map_estimate = spo.minimize(
    fun=posterior.evaluate_cost,
    jac=posterior.evaluate_gradient,
    x0=initial_guess,
    method="L-BFGS-B",
    options=optimizer_options,
)
print(map_estimate)
np.save("../results/map_estimate.npy", map_estimate.x)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 1018811.3689658777
        x: [-8.665e-20 -7.004e-20 ... -3.605e-15 -3.856e-20]
      nit: 1
      jac: [ 5.871e-04  4.746e-04 ...  2.443e+01  2.613e-04]
     nfev: 11
     njev: 11
 hess_inv: <15820x15820 LbfgsInvHessProduct with dtype=float64>


In [ ]:
map_parameter = np.load("map_estimate.npy")
analysis_data = analysis.compute_map_result_analysis(
    map_parameter=map_parameter,
    posterior=posterior,
    additional_output=additional_output,
)

In [ ]:
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_prior_mean,
    clim=[0, np.pi / 2]
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_ground_truth,
    clim=[0, np.pi / 2]
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.angle_map,
    clim=[0, np.pi / 2],
)

In [ ]:
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_angle_truth_prior,
    clim=[0, np.pi/ 2],
    circular=False,
)
visualization.visualize_full_scalar_field(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_angle_truth_map,
    clim=[0, np.pi / 2],
    circular=False,
)
visualization.visualize_arrival_times(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_prior,
)
visualization.visualize_arrival_times(
    mesh=additional_output.pv_mesh,
    scalar_field=analysis_data.diff_lat_truth_map,
)